# Exploring MOMENT — Time Series Foundation Model

This notebook walks through the basics of using [MOMENT](https://github.com/moment-timeseries-foundation-model/moment)
for industrial time-series analysis. MOMENT is a family of foundation models for general-purpose
time-series understanding — pre-trained on hundreds of thousands of time-series datasets.

We'll cover:
1. Installing and loading the model
2. Anomaly detection via reconstruction
3. Forecasting future values

**No training or fine-tuning required** — MOMENT works zero-shot on your data.

In [ ]:
# Install dependencies (run once)
!pip install momentfm torch numpy matplotlib

## Load Model

MOMENT uses a pipeline API similar to Hugging Face Transformers.
We'll load the `MOMENT-1-large` checkpoint.

In [ ]:
import numpy as np
import torch
from momentfm import MOMENTPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Generate synthetic vibration data (512 samples)
np.random.seed(42)
t = np.linspace(0, 4 * np.pi, 512)
signal = np.sin(t) + 0.5 * np.sin(3 * t) + np.random.normal(0, 0.1, 512)

# Inject anomaly at samples 350-380
signal[350:380] += np.random.normal(3.0, 0.5, 30)

print(f"Signal shape: {signal.shape}")
print(f"Signal range: [{signal.min():.2f}, {signal.max():.2f}]")

## Anomaly Detection

MOMENT's **reconstruction** task learns to reconstruct normal patterns.
High reconstruction error indicates anomalous regions.

In [ ]:
# Load MOMENT for reconstruction (anomaly detection)
model_recon = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large",
    model_kwargs={"task_name": "reconstruction"},
)
model_recon.init()
model_recon = model_recon.to(device)

# Prepare input: (batch, channels, seq_len)
x = torch.tensor(signal, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    output = model_recon(x)

reconstructed = output.output.squeeze().cpu().numpy()
recon_error = np.abs(signal - reconstructed)

# Threshold: mean + 3*std
threshold = recon_error.mean() + 3 * recon_error.std()
anomalies = np.where(recon_error > threshold)[0]

print(f"Reconstruction error — mean: {recon_error.mean():.4f}, std: {recon_error.std():.4f}")
print(f"Threshold: {threshold:.4f}")
print(f"Anomalous samples: {len(anomalies)}")
print(f"Anomaly indices: {anomalies.tolist()[:20]}")

# Plot
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(signal, label='Original', alpha=0.7)
axes[0].plot(reconstructed, label='Reconstructed', alpha=0.7)
axes[0].axvspan(350, 380, color='red', alpha=0.15, label='Injected anomaly')
axes[0].legend()
axes[0].set_title('Signal vs Reconstruction')

axes[1].plot(recon_error, color='orange', label='Reconstruction error')
axes[1].axhline(threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.2f})')
axes[1].legend()
axes[1].set_title('Reconstruction Error')

plt.tight_layout()
plt.show()

## Forecasting

MOMENT can predict future values given a historical window.
We'll forecast the next 96 time steps.

In [ ]:
# Load MOMENT for forecasting
forecast_horizon = 96
model_forecast = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large",
    model_kwargs={
        "task_name": "forecasting",
        "forecast_horizon": forecast_horizon,
    },
)
model_forecast.init()
model_forecast = model_forecast.to(device)

with torch.no_grad():
    forecast_output = model_forecast(x)

forecast = forecast_output.output.squeeze().cpu().numpy()

print(f"Forecast shape: {forecast.shape}")
print(f"Forecast range: [{forecast.min():.2f}, {forecast.max():.2f}]")

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(len(signal)), signal, label='Historical', alpha=0.7)
ax.plot(range(len(signal), len(signal) + forecast_horizon), forecast,
        label='Forecast', color='green', linewidth=2)
ax.axvline(len(signal), color='gray', linestyle='--', alpha=0.5)
ax.legend()
ax.set_title(f'MOMENT Forecast ({forecast_horizon} steps ahead)')
ax.set_xlabel('Sample index')
plt.tight_layout()
plt.show()

## Next Steps

- **Try with real data:** Load a CSV from `datasets/` or `samples/01-quick-test/sample_data.csv`
- **MQTT integration:** See `samples/02-mqtt-integration/` to test the full pipeline
- **End-to-end test:** Run `samples/03-end-to-end/run_e2e.py` against the Docker deployment
- **Use cases:** Read `docs/use-cases/` for predictive maintenance and anomaly detection scenarios

For architecture details, see [`docs/architecture.md`](../docs/architecture.md).